<a href="https://colab.research.google.com/github/siraz-kothiya/CVScanner/blob/main/CVScanner_HugginFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio sentence-transformers pdfplumber python-docx scikit-learn

In [ ]:
import gradio as gr
import pdfplumber
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

skills_db = ["python","java","c#","sql","aws","azure","docker","kubernetes",
             "tensorflow","pytorch","react","node.js","git","linux","javascript"]



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# File reading functions

def read_pdf(file):
    text = ""
    with pdfplumber.open(file.name) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

def read_docx(file):
    doc = Document(file.name)
    return "\n".join([p.text for p in doc.paragraphs])

def read_txt(file):
    return file.read().decode("utf-8")

def read_file(file):
    name = file.name.lower()
    if name.endswith(".pdf"):
        return read_pdf(file)
    elif name.endswith(".docx"):
        return read_docx(file)
    else:
        return read_txt(file)

In [ ]:
# Semantic Section Extraction
section_definitions = {
    "skills": "technical skills and technologies",
    "experience": "work experience and responsibilities",
    "projects": "projects and implementations",
    "education": "academic background and degrees"
}

# Precompute embeddings for sections
section_embeddings = {
    key: model.encode([value])[0] for key, value in section_definitions.items()
}


In [ ]:

def extract_sections_ai(text):
    # Split text into chunks (paragraphs)
    chunks = [c.strip() for c in text.split("\n") if len(c.strip()) > 30]
    sections = {key: "" for key in section_definitions.keys()}

    for chunk in chunks:
        chunk_emb = model.encode([chunk])[0]

        # Find the best matching section
        best_section = None
        best_score = -1
        for section, sec_emb in section_embeddings.items():
            score = cosine_similarity([chunk_emb], [sec_emb])[0][0]
            if score > best_score:
                best_score = score
                best_section = section

        sections[best_section] += chunk + " "

    return sections

# Skill extraction
def extract_skills(text):
    text = text.lower()
    return [skill for skill in skills_db if skill in text]

def compare_skills(cv_skills, jd_skills):
    matched = list(set(cv_skills) & set(jd_skills))
    missing = list(set(jd_skills) - set(cv_skills))
    score = len(matched)/max(len(jd_skills),1)
    return matched, missing, score


# Section similarity

def section_similarity(text1, text2):
    if len(text1.strip())==0 or len(text2.strip())==0:
        return 0
    emb1 = model.encode([text1])
    emb2 = model.encode([text2])
    return cosine_similarity(emb1, emb2)[0][0]


# Suggestions

def generate_suggestions(missing_skills, exp_score):
    suggestions = []
    if missing_skills:
        suggestions.append("Add experience or knowledge in: " + ", ".join(missing_skills))
    if exp_score < 0.6:
        suggestions.append("Highlight more relevant work experience for this job.")
    if not suggestions:
        suggestions.append("Your CV matches the job description well.")
    return suggestions

In [ ]:

# Main evaluation function
def evaluate(cv_file, jd_file):
    cv_text = read_file(cv_file)
    jd_text = read_file(jd_file)

    # Semantic section extraction
    cv_sections = extract_sections_ai(cv_text)
    jd_sections = extract_sections_ai(jd_text)

    # Compute similarity per section
    skills_sim = section_similarity(cv_sections["skills"], jd_sections["skills"])
    exp_sim = section_similarity(cv_sections["experience"], jd_sections["experience"])
    proj_sim = section_similarity(cv_sections["projects"], jd_sections["projects"])
    edu_sim = section_similarity(cv_sections["education"], jd_sections["education"])

    # Skill extraction & comparison
    cv_skills = extract_skills(cv_text)
    jd_skills = extract_skills(jd_text)
    matched, missing, skill_score = compare_skills(cv_skills, jd_skills)

    # Weighted overall score
    final_score = 0.4*skills_sim + 0.35*exp_sim + 0.15*proj_sim + 0.1*edu_sim
    final_score = round(final_score*100,2)

    # Suggestions
    suggestions = generate_suggestions(missing, exp_sim)

    # Result formatting
    result = f"""
    **CV Section**:{cv_sections}
    **JD Section**:{jd_sections}
✔️ **Overall Match Score:** {final_score}%

📊 **Section Match Scores**
Skills: {round(skills_sim*100,2)}%
Experience: {round(exp_sim*100,2)}%
Projects: {round(proj_sim*100,2)}%
Education: {round(edu_sim*100,2)}%

💡 **Matched Skills**
{', '.join(matched)}

❗ **Missing Skills**
{', '.join(missing)}

📝 **Suggestions**
{' ; '.join(suggestions)}
"""
    return result

In [ ]:
# ----------------------
# Gradio UI
# ----------------------
interface = gr.Interface(
    fn=evaluate,
    inputs=[
        gr.File(label="Upload CV (PDF/DOCX/TXT)"),
        gr.File(label="Upload Job Description")
    ],
    outputs=gr.Textbox(lines=20),
    title="AI-Powered CV Matcher",
    description="Upload resume and job description — AI will score and explain match."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://366a968dc21816d693.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
